# MINA completa en Google Colab

Este notebook enciende MINA desde Colab con **musica** y **voz IA con GPU**, sin depender de tu PC y sin Cloudflare para la voz.

## Uso rapido

1. En Colab activa GPU: `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU`.
2. Ejecuta `1) Instalar dependencias`. Al final Colab reinicia el runtime; eso es normal.
3. Ejecuta `2) Configurar token`.
4. Si cambiaste comandos, ejecuta `3) Registrar comandos`; si ya existen, puedes saltarlo.
5. Ejecuta `4) Encender MINA`.

El token no se guarda en GitHub. Puedes guardarlo en **Colab Secrets** como `DISCORD_TOKEN`, o pegarlo cuando la celda lo pida.

Si YouTube muestra `Sign in to confirm you're not a bot`, agrega en **Colab Secrets** uno de estos secretos:

- `YOUTUBE_COOKIES`: contenido completo de un archivo cookies.txt en formato Netscape.
- `YOUTUBE_COOKIES_B64`: el mismo archivo cookies.txt, pero codificado en Base64.

La celda de encendido crea automaticamente `/content/youtube_cookies.txt` y se lo pasa a `yt-dlp`.


In [ ]:
# 1) Instalar dependencias
import os
import subprocess
import sys

print('Instalando sistema: ffmpeg, curl, Node.js 22...')
subprocess.check_call('apt-get update -qq', shell=True)
subprocess.check_call('apt-get install -y -qq ffmpeg curl ca-certificates gnupg build-essential python3-dev', shell=True)
subprocess.check_call('curl -fsSL https://deb.nodesource.com/setup_22.x | bash -', shell=True)
subprocess.check_call('apt-get install -y -qq nodejs', shell=True)

print('Instalando Python: yt-dlp, ytmusicapi, Chatterbox...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'yt-dlp', 'ytmusicapi', 'chatterbox-tts', 'torchaudio',
])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
    'numpy==1.26.4', 'scipy', 'scikit-learn',
])
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchvision'])

print('Setup listo. Reiniciando runtime limpio para que NumPy/Chatterbox carguen bien...')
print('Si Colab muestra que la sesión falló o se desconectó, es normal: es el reinicio limpio.')
try:
    from google.colab import runtime
    restart_runtime = getattr(runtime, 'restart_runtime', None)
    if callable(restart_runtime):
        restart_runtime()
    else:
        raise AttributeError('runtime.restart_runtime no está disponible')
except Exception:
    os.kill(os.getpid(), 9)


In [ ]:
# 2) Configurar token y repo
import getpass
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/XDAVIXD01/mina-discord-music-bot.git'
REPO_DIR = Path('/content/mina-discord-music-bot')
DISCORD_CLIENT_ID = '1520888747292364900'
DISCORD_GUILD_ID = '753756346607861761'

token = os.environ.get('DISCORD_TOKEN', '').strip()
try:
    from google.colab import userdata
    token = token or userdata.get('DISCORD_TOKEN') or ''
except Exception:
    pass

if not token:
    token = getpass.getpass('Pega el DISCORD_TOKEN de MINA: ').strip()
if not token:
    raise RuntimeError('Falta DISCORD_TOKEN.')

os.environ['DISCORD_TOKEN'] = token
os.environ['DISCORD_CLIENT_ID'] = DISCORD_CLIENT_ID
os.environ['DISCORD_GUILD_ID'] = DISCORD_GUILD_ID

if REPO_DIR.exists():
    print('Actualizando repo...')
    subprocess.check_call(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
else:
    print('Clonando repo...')
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])

def run_logged(command, cwd=None, env=None):
    print('> ' + ' '.join(command))
    result = subprocess.run(command, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, command)

print('Instalando dependencias Node...')
run_logged(['npm', 'install', '--no-audit', '--no-fund', '--legacy-peer-deps'], cwd=REPO_DIR)
print('Compilando MINA sin activity web...')
run_logged(['npx', 'tsc'], cwd=REPO_DIR)

reference = REPO_DIR / 'assets' / 'voice' / 'user_reference.wav'
if not reference.exists():
    raise FileNotFoundError(f'No existe la referencia de voz: {reference}')

print('Listo. Repo:', REPO_DIR)
print('Referencia de voz:', reference, 'bytes:', reference.stat().st_size)


In [ ]:
# 3) Registrar comandos slash en tu servidor (opcional si ya están registrados)
import os
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/mina-discord-music-bot')
if not os.environ.get('DISCORD_TOKEN'):
    raise RuntimeError('Primero ejecuta la celda 2) Configurar token y repo.')

subprocess.check_call(['npm', 'run', 'deploy'], cwd=REPO_DIR, env=os.environ.copy())


In [ ]:
# 4) Encender MINA: musica + voz IA local en Colab
import base64
import os
import subprocess
import sys
import threading
from pathlib import Path

REPO_DIR = Path('/content/mina-discord-music-bot')
reference = REPO_DIR / 'assets' / 'voice' / 'user_reference.wav'
youtube_cookies_path = Path('/content/youtube_cookies.txt')

def _get_colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ''
    except Exception:
        return ''

def _prepare_youtube_cookies(env):
    raw_cookies = (env.get('YOUTUBE_COOKIES') or _get_colab_secret('YOUTUBE_COOKIES')).strip()
    b64_cookies = (env.get('YOUTUBE_COOKIES_B64') or _get_colab_secret('YOUTUBE_COOKIES_B64')).strip()
    if b64_cookies and not raw_cookies:
        raw_cookies = base64.b64decode(b64_cookies).decode('utf-8')
    if not raw_cookies:
        env.pop('YTDLP_COOKIES_FILE', None)
        print('YouTube cookies: no configuradas. Si YouTube bloquea musica, agrega YOUTUBE_COOKIES en Colab Secrets.')
        return
    ending = '\n' if not raw_cookies.endswith('\n') else ''
    youtube_cookies_path.write_text(raw_cookies + ending, encoding='utf-8')
    youtube_cookies_path.chmod(0o600)
    env['YTDLP_COOKIES_FILE'] = str(youtube_cookies_path)
    print('YouTube cookies: listas para yt-dlp.')

if 'MINA_PROCESS' in globals() and MINA_PROCESS.poll() is None:
    print('MINA ya esta encendida. PID:', MINA_PROCESS.pid)
else:
    env = os.environ.copy()
    env.update({
        'PYTHONIOENCODING': 'utf-8',
        'PYTHON_PATH': sys.executable,
        'VOICE_PYTHON_PATH': sys.executable,
        'VOICE_REFERENCE_PATH': str(reference),
        'VOICE_CLONE_ENABLED': '1',
        'VOICE_REMOTE_URL': '',
        'MINA_VOICE_MULTILINGUAL': '1',
        'YTDLP_PATH': 'yt-dlp',
        'FFMPEG_PATH': 'ffmpeg',
        'MINA_DISABLE_ACTIVITY': '1',
    })
    _prepare_youtube_cookies(env)
    if not env.get('DISCORD_TOKEN'):
        raise RuntimeError('Primero ejecuta la celda 2) Configurar token y repo.')
    MINA_PROCESS = subprocess.Popen(
        ['npm', 'start'],
        cwd=REPO_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    def _log():
        for line in MINA_PROCESS.stdout:
            print(line, end='')
    threading.Thread(target=_log, daemon=True).start()
    print('MINA encendida en Colab. PID:', MINA_PROCESS.pid)
    print('Usa /play para musica y /leer iniciar para voz IA.')


In [ ]:
# 5) Apagar MINA
if 'MINA_PROCESS' in globals() and MINA_PROCESS.poll() is None:
    MINA_PROCESS.terminate()
    print('MINA apagada.')
else:
    print('MINA no está encendida en esta sesión.')
